# Leverage Factor Analysis in Stock Market
This notebook constructs a leverage factor based on:
**Leverage = Total Assets / Equity**

The idea is to test whether more leveraged companies deliver higher or lower future returns.

## 1. Load and Prepare Financial Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load required data
assets = pd.read_csv('total_assets.csv', index_col=0, parse_dates=True)
equity = pd.read_csv('equity.csv', index_col=0, parse_dates=True)
announcement = pd.read_csv('announcement_date.csv', index_col=0)
market_cap = pd.read_csv('market_cap.csv', index_col=0, parse_dates=True)

# Format date indices
assets.index = assets.index.to_period('M').to_timestamp('M')
equity.index = equity.index.to_period('M').to_timestamp('M')
announcement.index = pd.to_datetime(announcement.index, errors='coerce')
announcement = announcement.applymap(lambda x: pd.to_datetime(x, errors='coerce'))


/var/folders/qv/t9wyzn094q3g9jdr0839rp7w0000gn/T/ipykernel_11924/319722419.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  announcement = announcement.applymap(lambda x: pd.to_datetime(x, errors='coerce'))


## 2. Calculate Leverage and Transform as Factor

In [2]:
# Handle duplicates and align frequency
assets = assets[~assets.index.duplicated(keep='last')]
equity = equity[~equity.index.duplicated(keep='last')]
assets = assets.reindex(market_cap.index).ffill()
equity = equity.reindex(market_cap.index).ffill()

# Compute Leverage
leverage = assets / equity
# Optionally use log transform to reduce skewness
leverage_factor = leverage.applymap(lambda x: np.log(x) if pd.notnull(x) and x > 0 else np.nan)

/var/folders/qv/t9wyzn094q3g9jdr0839rp7w0000gn/T/ipykernel_11924/2177438189.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  leverage_factor = leverage.applymap(lambda x: np.log(x) if pd.notnull(x) and x > 0 else np.nan)


## 3. Align Factor with Announcement Dates

In [3]:
def _deal_ann_date(anndt_bs, trade_days, stocks):
    matric_anndate_bs = {}
    for keyday in trade_days:
        matric_anndate_bs[keyday] = {}
        for ticker in stocks:
            try:
                bs = anndt_bs[ticker]
            except:
                continue
            report_date = -1
            for k, ann_dt in bs.items():
                if pd.notnull(ann_dt):
                    if ann_dt >= keyday:
                        break
                    else:
                        report_date = int(k.strftime('%Y%m%d'))
            matric_anndate_bs[keyday][ticker] = report_date
    return pd.DataFrame(matric_anndate_bs).T

def _date_adjust(df, matric_anndate):
    data = {}
    for c in df.columns:
        try:
            xd = matric_anndate[c]
            xd = xd.loc[xd > 0]
        except:
            continue
        v = df.loc[xd.values, c].values
        t = list(xd.index)
        data[c] = pd.Series(v, index=t)
    return pd.DataFrame(data)

# Apply to align leverage factor
matric_anndate = _deal_ann_date(announcement, market_cap.index, market_cap.columns)
leverage_factor_td = _date_adjust(leverage_factor, matric_anndate)

KeyError: "None of [Index([19700101, 19700101, 19700101, 19700101, 19700101, 19700101, 19700101,\n       19700101, 19700101, 19700101,\n       ...\n       19700101, 19700101, 19700101, 19700101, 19700101, 19700101, 19700101,\n       19700101, 19700101, 19700101],\n      dtype='int64', length=168)] are in the [index]"

## 4. Backtest: Top 50 Most Leveraged Stocks

In [ ]:
price = pd.read_csv('price_close_adj.csv', index_col=0, parse_dates=True)
returns = price.shift(-1) / price - 1

def top_n_fill(df, n):
    result_df = pd.DataFrame(0, index=df.index, columns=df.columns)
    for index, row in df.iterrows():
        top_indices = row.nlargest(n).index
        result_df.loc[index, top_indices] = 1
    return result_df

weights = top_n_fill(leverage_factor_td, 50)
portfolio_ret = (weights * returns).sum(1) / 50

## 5. Visualize Portfolio Performance

In [ ]:
portfolio_ret.index = [str(x) for x in portfolio_ret.index]
portfolio_ret.cumsum().plot(title='Cumulative Return: Leverage Factor Portfolio')
plt.ylabel('Cumulative Return')
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. IC (Information Coefficient) Analysis

In [ ]:
ic = leverage_factor_td.corrwith(returns, axis=1)
ic.resample('6M').mean().plot(marker='o', title='6-Month Average Information Coefficient')
plt.ylabel('IC')
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Turnover Analysis
Turnover measures how frequently the portfolio constituents change over time.
It is defined as the percentage of stocks replaced from one period to the next.

In [ ]:
# Turnover calculation
def calculate_turnover(weight_df):
    turnover = []
    prev = None
    for i in range(len(weight_df)):
        current = weight_df.iloc[i]
        if prev is not None:
            changed = (prev != current).sum()
            turnover.append(changed / (current.count() + 1e-6))  # avoid division by zero
        prev = current
    return pd.Series(turnover, index=weight_df.index[1:])

turnover_series = calculate_turnover(weights)
turnover_series.plot(title='Portfolio Turnover (Monthly)', figsize=(10, 4))
plt.ylabel('Turnover Rate')
plt.grid(True)
plt.tight_layout()
plt.show()